# Mancini Method: ES Futures Strategy Analysis

Interactive exploration and backtesting notebook for the Mancini long-only day trading strategy.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import date, datetime

from config.settings import (
    DEFAULT_STRATEGY, DEFAULT_ELEVATOR, DEFAULT_EXIT,
    DEFAULT_RISK, DEFAULT_SESSION, DEFAULT_CONTRACT
)
from core.market_data import MarketDataFetcher
from core.indicators import enrich_dataframe
from core.signals import SignalAggregator
from strategy.mancini_long import ManciniLongStrategy
from backtest.runner import BacktestRunner
from backtest.metrics import compute_metrics, format_metrics, monte_carlo_analysis
from backtest.visualizer import plot_trades, plot_equity_curve

print('Imports OK')

## 1. Load Data

Load ES futures data from Databento (or local cache/CSV).

In [ ]:
# Option A: Load from Databento (requires API key)
# fetcher = MarketDataFetcher()
# df = fetcher.get_single_day('ES', date(2024, 1, 15))

# Option B: Load from CSV
# df = MarketDataFetcher.load_csv('path/to/your/data.csv')

# Option C: Generate synthetic data for testing
from tests.conftest import make_selloff_then_recovery
df = make_selloff_then_recovery(
    start_price=5060.0,
    selloff_bars=12,
    selloff_rate=3.0,
    sweep_below=2.0,
    recovery_bars=6,
    hold_bars=10,
)
print(f'Loaded {len(df)} bars')
df.head()

## 2. Enrich with Indicators

In [ ]:
enriched = enrich_dataframe(df)
enriched[['close', 'atr_14', 'vwap', 'velocity_5', 'volume_spike']].tail(10)

## 3. Run Single-Day Strategy

In [ ]:
strategy = ManciniLongStrategy(min_rr_ratio=0.0)  # relax R:R for synthetic data
results = strategy.run_day(df)

print(f'Bars processed: {len(results)}')
print(f'Trades: {len(strategy.trade_records)}')
print(f'Total PnL: {strategy.total_pnl_pts:+.1f} pts (${strategy.total_pnl_dollars:+,.0f})')

for r in results:
    if r.signal is not None:
        print(f'  Signal @ bar {r.bar_idx}: {r.signal.signal_type.name} '
              f'entry={r.signal.entry_price:.2f} R:R={r.signal.rr_ratio_t1:.2f}')
    if r.entry_decision is not None and r.entry_decision.should_enter:
        print(f'  ENTRY @ bar {r.bar_idx}: {r.entry_decision.contracts} contracts')
    if r.exit_action is not None:
        print(f'  EXIT @ bar {r.bar_idx}: {r.exit_action.reason}')

## 4. Multi-Day Backtest

In [ ]:
# Example with synthetic multi-day data
daily_dfs = {}
for i in range(5):
    day = date(2024, 1, 15 + i)
    if day.weekday() < 5:
        daily_dfs[day] = make_selloff_then_recovery(
            start_price=5050 + np.random.randn() * 10,
            selloff_bars=10 + int(np.random.randn() * 2),
            selloff_rate=2.5 + np.random.rand(),
            sweep_below=1.0 + np.random.rand(),
            recovery_bars=5,
            hold_bars=8,
            start=datetime(day.year, day.month, day.day, 9, 30),
        )

runner = BacktestRunner(min_rr_ratio=0.0)
bt_result = runner.run_multi_day(daily_dfs=daily_dfs)

print(f'Days: {len(bt_result.days)}')
print(f'Total trades: {bt_result.total_trades}')
print(f'Total PnL: {bt_result.total_pnl_pts:+.1f} pts')

## 5. Performance Metrics

In [ ]:
metrics = compute_metrics(bt_result)
print(format_metrics(metrics))

## 6. Monte Carlo Analysis

In [ ]:
if bt_result.all_trades:
    mc = monte_carlo_analysis(bt_result.all_trades, n_simulations=10000, n_trades=100)
    print('Monte Carlo Results (10,000 simulations of 100 trades):')
    print(f'  Median PnL:     {mc["median_pnl"]:+.1f} pts')
    print(f'  5th percentile: {mc["p5_pnl"]:+.1f} pts')
    print(f'  95th percentile:{mc["p95_pnl"]:+.1f} pts')
    print(f'  P(profit):      {mc["prob_profit"]:.1%}')
    print(f'  Median Max DD:  {mc["max_dd_median"]:.1f} pts')
else:
    print('No trades to analyze')

## 7. Visualization

In [ ]:
# Plot a single day with trades
if bt_result.days:
    day0 = bt_result.days[0]
    day_df = list(daily_dfs.values())[0]
    # plot_trades(day0, day_df, backend='plotly')  # interactive
    print(f'Day: {day0.date}, Trades: {day0.num_trades}, PnL: {day0.pnl_pts:+.1f} pts')

In [ ]:
# Equity curve
if bt_result.all_trades:
    # plot_equity_curve(bt_result.all_trades)
    equity = np.cumsum([t.pnl_pts for t in bt_result.all_trades])
    print(f'Final equity: {equity[-1]:+.1f} pts')
    print(f'Peak equity:  {equity.max():+.1f} pts')